In [11]:
import pandas as pd
import numpy as np
import requests
import joblib
import os

from dotenv import load_dotenv

In [13]:
# Load environment variables

load_dotenv()

# Get API keys from environment variables
weather_api_key = os.getenv('OPENWEATHER_API_KEY')
ors_api_key = os.getenv('OPENROUTESERVICE_API_KEY')

print("Weather API Key Loaded:", weather_api_key is not None)
print("OpenRouteService API Key Loaded:", ors_api_key is not None)

Weather API Key Loaded: True
OpenRouteService API Key Loaded: True


In [14]:
# Load Trained Model

model=joblib.load("../outputs/saved_models/xgboost_model.pkl")
print("Model Loaded Successfully:", model is not None)

Model Loaded Successfully: True


In [15]:
# Test Weather API Integration

city = "Delhi"

url = (
    f"https://api.openweathermap.org/data/2.5/weather"
    f"?q={city}"
    f"&appid={weather_api_key}"
    f"&units=metric"
)

response = requests.get(url)

weather_data = response.json()

print(weather_data)

{'coord': {'lon': 77.2167, 'lat': 28.6667}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 44.05, 'feels_like': 42.71, 'temp_min': 44.05, 'temp_max': 44.05, 'pressure': 996, 'humidity': 14, 'sea_level': 996, 'grnd_level': 972}, 'visibility': 9000, 'wind': {'speed': 4.12, 'deg': 240}, 'clouds': {'all': 20}, 'dt': 1779272324, 'sys': {'type': 1, 'id': 9165, 'country': 'IN', 'sunrise': 1779235077, 'sunset': 1779284256}, 'timezone': 19800, 'id': 1273294, 'name': 'Delhi', 'cod': 200}


In [17]:
# Extract relevant weather features for prediction

weather_condition=weather_data['weather'][0]['main'].lower()
temperature=weather_data['main']['temp']

print(f"Current Weather in {city}: {weather_condition}, {temperature}°C")

Current Weather in Delhi: clouds, 44.05°C


In [18]:
# Normalize weather condition for model input

if weather_condition in [
    "rain",
    "drizzle",
    "thunderstorm"
]:
    normalized_weather = "rain"

elif weather_condition in [
    "snow"
]:
    normalized_weather = "snow"

else:
    normalized_weather = "clear"

print("Normalized Weather:", normalized_weather)

Normalized Weather: clear


In [19]:
# Test OpenRouteService API Integration

start_location= "Delhi"
end_location= "Gurgaon"

geocode_url = "https://api.openrouteservice.org/geocode/search"

headers = {
    "Authorization": ors_api_key
}

# Get coordinates for start location
start_response = requests.get(
    geocode_url,
    headers=headers,
    params={"text": start_location}
)

start_data = start_response.json()

# Get coordinates for end location
end_response = requests.get(
    geocode_url,
    headers=headers,
    params={"text": end_location}
)

end_data = end_response.json()

# Extract coordinates
start_coords = start_data["features"][0]["geometry"]["coordinates"]
end_coords = end_data["features"][0]["geometry"]["coordinates"]

print("Start Coordinates:", start_coords)
print("End Coordinates:", end_coords)

Start Coordinates: [77.163665, 28.557163]
End Coordinates: [77.061922, 28.459647]


In [20]:
# Fetch route information

route_url = "https://api.openrouteservice.org/v2/directions/driving-car"

headers = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json"
}

body = {
    "coordinates": [
        start_coords,
        end_coords
    ]
}

route_response = requests.post(
    route_url,
    json=body,
    headers=headers
)

route_data = route_response.json()

print(route_data)

{'bbox': [77.059769, 28.459528, 77.16569, 28.55711], 'routes': [{'summary': {'distance': 19991.0, 'duration': 1414.0}, 'segments': [{'distance': 19991.0, 'duration': 1414.0, 'steps': [{'distance': 508.4, 'duration': 44.7, 'type': 11, 'instruction': 'Head southeast on Munirka Marg', 'name': 'Munirka Marg', 'way_points': [0, 7]}, {'distance': 3145.9, 'duration': 203.2, 'type': 1, 'instruction': 'Turn right onto Nelson Mandela Marg', 'name': 'Nelson Mandela Marg', 'way_points': [7, 35]}, {'distance': 3137.1, 'duration': 188.2, 'type': 1, 'instruction': 'Turn right onto Abdul Gaffar Khan Marg', 'name': 'Abdul Gaffar Khan Marg', 'way_points': [35, 99]}, {'distance': 1289.6, 'duration': 91.7, 'type': 12, 'instruction': 'Keep left', 'name': '-', 'way_points': [99, 124]}, {'distance': 3225.0, 'duration': 174.3, 'type': 13, 'instruction': 'Keep right', 'name': '-', 'way_points': [124, 144]}, {'distance': 6164.8, 'duration': 400.1, 'type': 12, 'instruction': 'Keep left', 'name': '-', 'way_points

In [21]:
# Extract distance and duration

summary = route_data["routes"][0]["summary"]

# Distance in kilometers
distance_km = summary["distance"] / 1000

# Duration in minutes
duration_minutes = summary["duration"] / 60

print(f"Distance: {distance_km:.2f} km")
print(f"Estimated Duration: {duration_minutes:.2f} minutes")

Distance: 19.99 km
Estimated Duration: 23.57 minutes


In [22]:
# Estimate traffic conditions

average_speed= distance_km / (duration_minutes / 60)

if (average_speed > 40):
    traffic = "low"

elif (average_speed > 20):
    traffic = "medium"

else:
    traffic = "high"

print(f"Estimated Traffic Condition: {traffic} (Average Speed: {average_speed:.2f} km/h)")

Estimated Traffic Condition: low (Average Speed: 50.90 km/h)


In [23]:
# Weather fetch function

def get_weather(city):

    url = (
        f"https://api.openweathermap.org/data/2.5/weather"
        f"?q={city}"
        f"&appid={weather_api_key}"
        f"&units=metric"
    )

    response = requests.get(url)

    weather_data = response.json()

    # Extract weather condition
    weather_condition = (
        weather_data["weather"][0]["main"].lower()
    )

    # Extract temperature
    temperature = weather_data["main"]["temp"]

    # Normalize weather
    if weather_condition in [
        "rain",
        "drizzle",
        "thunderstorm"
    ]:
        normalized_weather = "rain"

    elif weather_condition in ["snow"]:
        normalized_weather = "snow"

    else:
        normalized_weather = "clear"

    return {
        "weather": normalized_weather,
        "temperature": temperature
    }

In [24]:
# Route + Traffic fetch function

def get_route_info(start_location, end_location):

    geocode_url = (
        "https://api.openrouteservice.org/geocode/search"
    )

    headers = {
        "Authorization": ors_api_key
    }

    # Start location
    start_response = requests.get(
        geocode_url,
        headers=headers,
        params={"text": start_location}
    )

    start_data = start_response.json()

    # End location
    end_response = requests.get(
        geocode_url,
        headers=headers,
        params={"text": end_location}
    )

    end_data = end_response.json()

    # Extract coordinates
    start_coords = (
        start_data["features"][0]["geometry"]["coordinates"]
    )

    end_coords = (
        end_data["features"][0]["geometry"]["coordinates"]
    )


    route_url = (
        "https://api.openrouteservice.org/v2/directions/driving-car"
    )

    headers = {
        "Authorization": ors_api_key,
        "Content-Type": "application/json"
    }

    body = {
        "coordinates": [
            start_coords,
            end_coords
        ]
    }

    route_response = requests.post(
        route_url,
        json=body,
        headers=headers
    )

    route_data = route_response.json()

 
    summary = route_data["routes"][0]["summary"]

    distance_km = summary["distance"] / 1000
    duration_minutes = summary["duration"] / 60


    average_speed = (
        distance_km /
        (duration_minutes / 60)
    )

    if average_speed > 40:
        traffic = "low"

    elif average_speed > 20:
        traffic = "medium"

    else:
        traffic = "high"

    return {
        "distance_km": distance_km,
        "duration_minutes": duration_minutes,
        "traffic": traffic
    }

In [25]:
# FEATURE GENERATION FUNCTION

def create_features(distance, hour, traffic, weather):

    # Peak and night flags
    is_peak = 1 if 7 <= hour <= 10 or 17 <= hour <= 21 else 0
    is_night = 1 if hour >= 22 or hour <= 5 else 0

    # Cyclical hour encoding
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    # Encoding maps
    traffic_map = {
        'low': 1,
        'medium': 2,
        'high': 3
    }

    weather_map = {
        'clear': 1,
        'rain': 2,
        'snow': 3
    }

    # Encode categorical inputs
    traffic_encoded = traffic_map[traffic]
    weather_encoded = weather_map[weather]

    # Distance transformations
    distance_squared = distance ** 2
    distance_log = np.log1p(distance)

    # Distance bucket
    if distance <= 5:
        distance_bucket = 0
    elif distance <= 12:
        distance_bucket = 1
    elif distance <= 25:
        distance_bucket = 2
    elif distance <= 35:
        distance_bucket = 3
    else:
        distance_bucket = 4

    # Interaction features
    distance_traffic = distance * traffic_encoded
    distance_peak = distance * is_peak
    traffic_peak = traffic_encoded * is_peak
    weather_peak = weather_encoded * is_peak

    # Demand score
    demand_score = (
        is_peak * 2 +
        weather_encoded * 1.5 +
        traffic_encoded * 1.2
    )

    # Create dataframe
    features = pd.DataFrame([{
        "distance": distance,
        "hour": hour,
        "is_peak": is_peak,
        "is_night": is_night,
        "hour_sin": hour_sin,
        "hour_cos": hour_cos,
        "traffic_encoded": traffic_encoded,
        "weather_encoded": weather_encoded,
        "distance_squared": distance_squared,
        "distance_log": distance_log,
        "distance_bucket": distance_bucket,
        "distance_traffic": distance_traffic,
        "distance_peak": distance_peak,
        "traffic_peak": traffic_peak,
        "weather_peak": weather_peak,
        "demand_score": demand_score
    }])

    return features

In [28]:
# Live Prediction Function

def predict_fare(start_location, end_location, city, hour):

    # Get route and traffic info
    route_info = get_route_info(start_location, end_location)
    distance=route_info["distance_km"]
    traffic=route_info["traffic"]

    # Get weather info (using start location city for simplicity)
    weather_info = get_weather(city)
    weather=weather_info["weather"]

    # Create features
    features = create_features(
        distance=distance,
        hour=hour,
        traffic=traffic,
        weather=weather
    )

    # Predict fare
    predicted_fare = model.predict(features)[0]

    return {
        "predicted_fare": float(round(predicted_fare, 2)),
        "distance_km": float(round(distance, 2)),
        "traffic": traffic,
        "weather": weather
    }

In [29]:
# Testing the live prediction function

result = predict_fare(
    start_location="Delhi",
    end_location="Gurgaon",
    city="Delhi",
    hour=9
)

print("Live Prediction Result:", result)

Live Prediction Result: {'predicted_fare': 675.5, 'distance_km': 19.99, 'traffic': 'low', 'weather': 'clear'}
